<a href="https://colab.research.google.com/github/nukui96/mackenzie-pa4-bss/blob/main/notebooks/02_cd_projeto_aplicado_IV_doc_etapa2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir este notebook no Colab"/></a>

<img src="https://raw.githubusercontent.com/scalabrinig/cdProjetoAplicadoIV/d093146488f56dfcf0ef286bcee8efe0e71b9c76/figuras/mackenzie_logo.jpg" width="25%" align="right"/>

# **PROJETO APLICADO IV - Tecnologia em Banco de Dados - 2026/02**

# **Documentação do Projeto**

---

> **Etapa 2:** Referencial Teórico, Pipeline da Solução e Cronograma (Marco: 28/09/2026)

# **Título do Projeto**

## Previsão Diária de Demanda de Bicicletas Compartilhadas (London Bike Sharing)

---

In [1]:
#@title **Identificação do Grupo**

#@markdown Integrantes do Grupo, nome completo em ordem alfabética (*informe: <nome>, <matrícula>*)
Aluno1 = 'Kayo Oliveira Nukui, 10356420' #@param {type:"string"}

# **Resumo da etapa 2**

Este projeto investiga a previsão do total diário de aluguéis de bicicletas em Londres para o dia seguinte. Esta versão reúne a proposta, um referencial teórico, o pipeline planejado, o cronograma e a descrição da base de dados. A base tem 17.414 registros horários distribuídos por 730 datas observadas em um intervalo de 731 dias calendário. As técnicas de previsão e sua avaliação são tratadas como próximas entregas acadêmicas; não há resultados preditivos neste caderno.

# **Introdução**

Este estudo propõe prever o total diário de aluguéis de bicicletas em Londres para o dia seguinte. Uma previsão suficientemente precisa poderia ser útil ao planejamento agregado, mas essa utilidade ainda não foi demonstrada. A pergunta de pesquisa é se a suavização exponencial de Holt-Winters e uma regressão Ridge com defasagens superam uma referência sazonal semanal em avaliação temporal sem vazamento.

O tema pode ser relacionado ao **ODS 11 – Cidades e Comunidades Sustentáveis**, especialmente à meta 11.2 (UNITED NATIONS, s.d.). Essa relação é temática: os dados históricos não permitem afirmar redução de emissões ou impacto real na mobilidade. O *London Bike Sharing Dataset* contém 17.414 registros horários de 04/01/2015 a 03/01/2017, com demanda (`cnt`), clima e indicadores de calendário.


# **Referencial Teórico**

Séries temporais podem apresentar dependência entre observações, tendência e sazonalidade (BOX et al., 2016; HYNDMAN; ATHANASOPOULOS, 2021). Neste projeto, a existência e a utilidade de um ciclo semanal na demanda diária serão examinadas, não presumidas.

**Trabalhos correlatos.** Beecham, Wood e Bowerman (2014) analisaram viagens origem-destino do sistema londrino de bicicletas e identificaram padrões de deslocamento pendular por análise espacial e visual. A unidade de análise deles é distinta da demanda diária agregada deste estudo. Gebhart e Noland (2014) examinaram viagens do sistema de Washington, DC, e encontraram menor uso e duração sob frio, chuva e alta umidade. O estudo não avaliou os modelos de previsão aqui propostos. Esses trabalhos motivam investigar calendário e clima, mas não comprovam o desempenho de previsões nesta base. Shi et al. (2023) compararam LSTM e outros modelos para demanda horária em Londres, abordagem de granularidade e complexidade diferentes da previsão diária proposta aqui.

**Alternativas propostas.** A referência *Seasonal Naive* repete a observação de sete dias antes: para prever $t+1$ com dados até $t$, usa-se $\hat{y}_{t+1}=y_{t-6}$ (HYNDMAN; ATHANASOPOULOS, 2021). É simples e interpretável, mas pode falhar em feriados e mudanças súbitas. A suavização de Holt-Winters representa nível, tendência e sazonalidade, com período semanal como hipótese a testar (HOLT, 1957; WINTERS, 1960); sua formulação clássica não usa diretamente clima ou calendário. A regressão Ridge penaliza coeficientes por $L_2$ e permite combinar defasagens e calendário (HASTIE; TIBSHIRANI; FRIEDMAN, 2009), mas requer preparação cuidadosa para impedir vazamento temporal. A comparação das técnicas pertence às próximas etapas; clima observado no dia previsto não será usado como entrada de uma previsão feita na véspera.


# **Pipeline proposto**

```text
CSV horário original
    ↓
Conferência de colunas, datas, duplicatas e cobertura horária
    ↓
Soma de cnt por dia observado + marcação explícita de dias sem registros
    ↓
Etapa 3: análise exploratória e pré-processamento
    ↓
Etapas 3 e 4: decisão documentada sobre dias incompletos → divisão cronológica →
linha de base sazonal e modelos candidatos → MAE/RMSE em períodos futuros
```

A unidade de análise proposta é o dia civil, mas o dia 02/09/2016 não tem registros no CSV. A ausência de registros em uma data exige investigação antes de definir o alvo diário. A regra de tratamento dos dias incompletos será justificada na etapa de implementação parcial.

Para prever o dia seguinte, as entradas deverão estar disponíveis até o fechamento do dia anterior. Temperatura e demanda observadas no próprio dia previsto não serão usadas como entradas. O recorte de treino, validação e teste será fixado depois da decisão de qualidade da série, sem embaralhar datas.

### Cronograma de atividades

| Etapa | Prazo oficial | Marcos de trabalho | Estado acadêmico |
|---|---|---|---|
| 1 | 31/08/2026 | Delimitar problema, base, objetivo e proposta | Arquivos locais; submissão a confirmar |
| 2 | 28/09/2026 | Conferir fontes, descrever pipeline, auditar dados, caracterizar a série e revisar a entrega | Pacote local revisado; submissão a confirmar |
| 3 | 26/10/2026 | Até 05/10: decidir lacunas; 12/10: fixar cortes; 19/10: ajustar referência e primeiro modelo; 26/10: avaliar e documentar | Entrega acadêmica futura |
| 4 | 30/11/2026 | Até 09/11: segunda técnica; 16/11: comparação; 23/11: discussão e conclusões; 30/11: revisão final e apresentação | Entrega acadêmica futura |

As datas intermediárias são marcos planejados pelo autor; os prazos de cada etapa vêm do roteiro da disciplina. O cronograma será atualizado conforme o trabalho e as orientações recebidas.


# **EDA e Pré-processamento dos dados**

### Inspeção Inicial dos Dados Brutos (Etapa 1)
Carregamos o arquivo original da TfL (`london_merged.csv`) para verificar dimensões, tipos de colunas, primeiras linhas e integridade (presença de valores nulos).

In [ ]:
import pandas as pd
import numpy as np
import os

# Localização do arquivo de dados brutos
caminho_dados = 'data/raw/london_merged.csv'
if not os.path.exists(caminho_dados):
    caminho_dados = '../data/raw/london_merged.csv'

# Leitura da base original
df = pd.read_csv(caminho_dados)

print(f"Dimensões da base: {df.shape[0]} linhas (registros horários) e {df.shape[1]} colunas.")
print("\nPrimeiras 5 linhas da base:")
df.head()

Dimensões da base: 17414 linhas (registros horários) e 10 colunas.

Primeiras 5 linhas da base:


,timestamp,cnt,t1,t2,hum,wind_speed,weather_code,is_holiday,is_weekend,season
0,2015-01-04 00:00:00,182,3.0,2.0,93.0,6.0,3.0,0.0,1.0,3.0
1,2015-01-04 01:00:00,138,3.0,2.5,93.0,5.0,1.0,0.0,1.0,3.0
2,2015-01-04 02:00:00,134,2.5,2.5,96.5,0.0,1.0,0.0,1.0,3.0
3,2015-01-04 03:00:00,72,2.0,2.0,100.0,0.0,1.0,0.0,1.0,3.0
4,2015-01-04 04:00:00,47,2.0,0.0,93.0,6.5,1.0,0.0,1.0,3.0


In [ ]:
# Verificação das colunas, tipos e valores ausentes
print("Informações estruturais e tipos:")
df.info()

print("\nContagem de valores ausentes (nulos) por coluna:")
print(df.isnull().sum())

Informações estruturais e tipos:
<class 'pandas.DataFrame'>
RangeIndex: 17414 entries, 0 to 17413
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   timestamp     17414 non-null  str    
 1   cnt           17414 non-null  int64  
 2   t1            17414 non-null  float64
 3   t2            17414 non-null  float64
 4   hum           17414 non-null  float64
 5   wind_speed    17414 non-null  float64
 6   weather_code  17414 non-null  float64
 7   is_holiday    17414 non-null  float64
 8   is_weekend    17414 non-null  float64
 9   season        17414 non-null  float64
dtypes: float64(8), int64(1), str(1)
memory usage: 1.3 MB

Contagem de valores ausentes (nulos) por coluna:
timestamp       0
cnt             0
t1              0
t2              0
hum             0
wind_speed      0
weather_code    0
is_holiday      0
is_weekend      0
season          0
dtype: int64


In [ ]:
# Resumo estatístico das variáveis numéricas
print("Estatísticas descritivas básicas:")
df[['cnt', 't1', 't2', 'hum', 'wind_speed']].describe()

Estatísticas descritivas básicas:


,cnt,t1,t2,hum,wind_speed
count,17414.000000,17414.000000,17414.000000,17414.000000,17414.000000
mean,1143.101642,12.468091,11.520836,72.324954,15.913063
std,1085.108068,5.571818,6.615145,14.313186,7.894570
min,0.000000,-1.500000,-6.000000,20.500000,0.000000
25%,257.000000,8.000000,6.000000,63.000000,10.000000
50%,844.000000,12.500000,12.500000,74.500000,15.000000
75%,1671.750000,16.000000,16.000000,83.000000,20.500000
max,7860.000000,34.000000,34.000000,100.000000,56.500000


### Limitações e observações iniciais

Os registros são horários, enquanto o objetivo é uma previsão diária. A agregação exigirá atenção à cobertura de cada data. Entre 04/01/2015 e 03/01/2017 há 731 dias calendário; uma grade simples de 24 posições por dia teria 17.544 posições, contra 17.414 registros presentes. A diferença de 130 posições não equivale automaticamente a falhas de coleta: o arquivo não declara fuso horário e uma transição de horário civil pode alterar a duração de um dia. As lacunas e os dias incompletos serão examinados antes de definir o alvo da modelagem.


# **Modelos, resultados e conclusão**

Seções previstas pelo template oficial para as próximas entregas acadêmicas. Este notebook da etapa 2 não apresenta análise exploratória aprofundada, modelos treinados nem resultados de previsão.

# **Referências**

* BEECHAM, R.; WOOD, J.; BOWERMAN, A. Studying commuting behaviours using collaborative visual analytics. **Computers, Environment and Urban Systems**, v. 47, p. 5-15, 2014. DOI: 10.1016/j.compenvurbsys.2013.10.007.
* BOX, G. E. P.; JENKINS, G. M.; REINSEL, G. C.; LJUNG, G. M. **Time Series Analysis: Forecasting and Control**. 5. ed. Hoboken: John Wiley & Sons, 2016.
* GEBHART, K.; NOLAND, R. B. The impact of weather conditions on bikeshare trips in Washington, DC. **Transportation**, v. 41, p. 1205-1225, 2014. DOI: 10.1007/s11116-014-9540-7.
* HASTIE, T.; TIBSHIRANI, R.; FRIEDMAN, J. **The Elements of Statistical Learning: Data Mining, Inference, and Prediction**. 2. ed. New York: Springer, 2009.
* HOLT, C. C. Forecasting seasonals and trends by exponentially weighted moving averages. **Office of Naval Research Research Memorandum**, n. 52, 1957.
* HYNDMAN, R. J.; ATHANASOPOULOS, G. **Forecasting: Principles and Practice**. 3. ed. Melbourne: OTexts, 2021. Disponível em: <https://otexts.com/fpp3/>. Acesso em: 28 set. 2026.
* MAVRODIEV, H. **London Bike Sharing Dataset**. Kaggle, [s.d.]. Disponível em: <https://www.kaggle.com/datasets/hmavrodiev/london-bike-sharing-dataset>. Acesso em: 28 set. 2026.
* SHI, Y.; ZHANG, L.; LU, S.; LIU, Q. Short-Term Demand Prediction of Shared Bikes Based on LSTM Network. **Electronics**, v. 12, n. 6, art. 1381, 2023. DOI: 10.3390/electronics12061381.
* TRANSPORT FOR LONDON (TfL). **Open data users**. [s.d.]. Disponível em: <https://tfl.gov.uk/info-for/open-data-users/>. Acesso em: 28 set. 2026.
* UNITED NATIONS. **Goal 11: Sustainable Cities and Communities**. Department of Economic and Social Affairs, [s.d.]. Disponível em: <https://sdgs.un.org/goals/goal11>. Acesso em: 28 set. 2026.
* WINTERS, P. R. Forecasting sales by exponentially weighted moving averages. **Management Science**, v. 6, n. 3, p. 324-342, 1960.
